# Train BBDM on Paired PNG Data

This notebook trains the modified BBDM pipeline for paired PNG virtual staining data:

- input: grayscale PNG, 512 x 512
- target: RGB PNG, 512 x 512
- training crop: 256 x 256
- condition encoder: 1 channel -> 3 channels

Run this notebook in a Colab GPU runtime. Edit the paths in the config cell before running.

In [ ]:
# Check GPU. In Colab, use Runtime -> Change runtime type -> GPU.
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

Update `PROJECT_DRIVE_DIR` and `DATASET_ZIP` if your files are stored in a different Drive path.

In [ ]:
from pathlib import Path

PROJECT_DRIVE_DIR = Path('/content/drive/MyDrive/Super-resolved-virtual-staining')
DATASET_ZIP = PROJECT_DRIVE_DIR / 'test-20260515T193153Z-3-001.zip'

WORKDIR = Path('/content/Super-resolved-virtual-staining')
DATA_ROOT = Path('/content/data')
OUTPUT_DRIVE_DIR = Path('/content/drive/MyDrive/bbdm_outputs')

CROP_SIZE = 256
BATCH_SIZE = 1

print('Project source:', PROJECT_DRIVE_DIR)
print('Dataset zip:', DATASET_ZIP)
print('Runtime workdir:', WORKDIR)

## Install Dependencies

In [ ]:
!apt-get -qq update
!apt-get -qq install -y libopenmpi-dev openmpi-bin
%pip -q install blobfile mpi4py pytorch-wavelets lpips scikit-image opencv-python tqdm

## Copy Project to Colab Runtime

Copying the project from Drive to `/content` makes training faster and avoids writing many temporary files to Drive.

In [ ]:
import os
import shutil
import sys

assert PROJECT_DRIVE_DIR.exists(), f'Project path not found: {PROJECT_DRIVE_DIR}'

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

ignore = shutil.ignore_patterns('.git', '__pycache__', '*.pyc', '*.zip')
shutil.copytree(PROJECT_DRIVE_DIR, WORKDIR, ignore=ignore)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))

print('Current directory:', os.getcwd())

## Extract Dataset

The original zip in Drive is not modified. Files are extracted into Colab runtime storage.

In [ ]:
import zipfile

assert DATASET_ZIP.exists(), f'Dataset zip not found: {DATASET_ZIP}'

if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(DATASET_ZIP) as zf:
    zf.extractall(DATA_ROOT)

INPUT_DIR = DATA_ROOT / 'test' / 'input'
TARGET_DIR = DATA_ROOT / 'test' / 'target'

assert INPUT_DIR.exists(), f'Input directory not found: {INPUT_DIR}'
assert TARGET_DIR.exists(), f'Target directory not found: {TARGET_DIR}'
print('Input dir:', INPUT_DIR)
print('Target dir:', TARGET_DIR)

## Inspect Dataset

In [ ]:
from PIL import Image

input_files = sorted(INPUT_DIR.glob('*.png'))
target_files = sorted(TARGET_DIR.glob('*.png'))
input_names = {p.name for p in input_files}
target_names = {p.name for p in target_files}
common_names = sorted(input_names & target_names)

print('input_count:', len(input_files))
print('target_count:', len(target_files))
print('paired_count:', len(common_names))
print('only_input:', len(input_names - target_names))
print('only_target:', len(target_names - input_names))
assert common_names, 'No matching input-target filenames found.'

for name in common_names[:5]:
    inp = Image.open(INPUT_DIR / name)
    tgt = Image.open(TARGET_DIR / name)
    print(name, 'input', inp.mode, inp.size, '| target', tgt.mode, tgt.size)

## Smoke Test Loader and Condition Encoder

This checks the modified PNG data path before starting training.

In [ ]:
import torch
from improved_diffusion.image_datasets import load_paired_png_data
from improved_diffusion.unet import ConditionEncoder

data = load_paired_png_data(
    input_dir=str(INPUT_DIR),
    target_dir=str(TARGET_DIR),
    batch_size=2,
    image_size=CROP_SIZE,
    deterministic=True,
)
target, cond, kwargs = next(data)
print('target:', tuple(target.shape), float(target.min()), float(target.max()))
print('cond:', tuple(cond.shape), float(cond.mean()), float(cond.std()))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = ConditionEncoder(1, 3).to(device).eval()
with torch.no_grad():
    encoded = encoder(cond.to(device).float())
print('encoded:', tuple(encoded.shape), float(encoded.min()), float(encoded.max()))
assert target.shape[1:] == encoded.shape[1:], 'Target and encoded condition shapes must match.'

## Optional: One Diffusion Loss Check

This creates a very small diffusion model and computes one loss. It is slower than the loader test but catches shape errors in the BBDM core.

In [ ]:
from improved_diffusion.script_util import sr_create_model_and_diffusion

small_model, small_diffusion = sr_create_model_and_diffusion(
    large_size=CROP_SIZE,
    small_size=CROP_SIZE,
    class_cond=False,
    learn_sigma=False,
    num_channels=32,
    num_res_blocks=1,
    num_heads=4,
    num_heads_upsample=-1,
    attention_resolutions='16,8',
    dropout=0.0,
    diffusion_steps=10,
    noise_schedule='linear',
    timestep_respacing='',
    use_kl=False,
    predict_xstart=False,
    rescale_timesteps=True,
    rescale_learned_sigmas=True,
    use_checkpoint=False,
    use_scale_shift_norm=True,
    in_channels=3,
    out_channels=3,
)
small_model = small_model.to(device).eval()
t = torch.randint(0, small_diffusion.num_timesteps, (target.shape[0],), device=device)
with torch.no_grad():
    loss_terms = small_diffusion.training_losses(
        small_model,
        target.to(device).float(),
        encoded.float(),
        t,
    )
print({k: float(v.mean().detach().cpu()) for k, v in loss_terms.items()})

## Smoke Train

This short run verifies the full training loop. It is not meant to produce good image quality.

In [ ]:
import subprocess

SMOKE_MODEL_DIR = Path('/content/bbdm_smoke_models')
SMOKE_LOG_DIR = Path('/content/bbdm_smoke_log')
for path in [SMOKE_MODEL_DIR, SMOKE_LOG_DIR]:
    if path.exists():
        shutil.rmtree(path)

cmd = [
    sys.executable, 'train.py.py',
    '--lr_data_dir', str(INPUT_DIR),
    '--hr_data_dir', str(TARGET_DIR),
    '--batch_size', str(BATCH_SIZE),
    '--large_size', str(CROP_SIZE),
    '--small_size', str(CROP_SIZE),
    '--num_channels', '64',
    '--num_res_blocks', '1',
    '--diffusion_steps', '100',
    '--lr_anneal_steps', '20',
    '--log_interval', '1',
    '--val_interval', '10',
    '--save_interval', '20',
    '--model_dir', str(SMOKE_MODEL_DIR),
    '--log_dir', str(SMOKE_LOG_DIR),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## Visualize Smoke Train Outputs

The training loop writes validation examples as `*_LR.png`, `*_SR.png`, and `*_HR.png` in the log directory.

In [ ]:
import matplotlib.pyplot as plt

def show_validation_triplets(log_dir, max_items=4):
    log_dir = Path(log_dir)
    triplets = []
    for idx in range(max_items):
        lr = log_dir / f'{idx}_LR.png'
        sr = log_dir / f'{idx}_SR.png'
        hr = log_dir / f'{idx}_HR.png'
        if lr.exists() and sr.exists() and hr.exists():
            triplets.append((lr, sr, hr))
    if not triplets:
        print('No validation images found in', log_dir)
        return

    fig, axes = plt.subplots(len(triplets), 3, figsize=(12, 4 * len(triplets)))
    if len(triplets) == 1:
        axes = [axes]
    for row, (lr, sr, hr) in zip(axes, triplets):
        for ax, path, title in zip(row, [lr, sr, hr], ['Condition', 'Prediction', 'Target']):
            ax.imshow(Image.open(path))
            ax.set_title(title)
            ax.axis('off')
    plt.tight_layout()
    plt.show()

show_validation_triplets(SMOKE_LOG_DIR)

## Longer Train

Set `RUN_LONG_TRAIN = True` only after the smoke train finishes successfully. Adjust `LONG_STEPS` based on your GPU time.

In [ ]:
RUN_LONG_TRAIN = False
LONG_STEPS = 5000
LONG_MODEL_DIR = Path('/content/bbdm_long_models')
LONG_LOG_DIR = Path('/content/bbdm_long_log')

if RUN_LONG_TRAIN:
    for path in [LONG_MODEL_DIR, LONG_LOG_DIR]:
        if path.exists():
            shutil.rmtree(path)
    cmd = [
        sys.executable, 'train.py.py',
        '--lr_data_dir', str(INPUT_DIR),
        '--hr_data_dir', str(TARGET_DIR),
        '--batch_size', str(BATCH_SIZE),
        '--large_size', str(CROP_SIZE),
        '--small_size', str(CROP_SIZE),
        '--num_channels', '128',
        '--num_res_blocks', '2',
        '--diffusion_steps', '1000',
        '--lr_anneal_steps', str(LONG_STEPS),
        '--log_interval', '10',
        '--val_interval', '500',
        '--save_interval', '1000',
        '--model_dir', str(LONG_MODEL_DIR),
        '--log_dir', str(LONG_LOG_DIR),
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Long train skipped. Set RUN_LONG_TRAIN = True to run it.')

## Save Models and Logs to Drive

In [ ]:
OUTPUT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

for src in [SMOKE_MODEL_DIR, SMOKE_LOG_DIR, Path('/content/bbdm_long_models'), Path('/content/bbdm_long_log')]:
    if src.exists():
        dst = OUTPUT_DRIVE_DIR / src.name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print('Copied', src, '->', dst)

print('Output Drive dir:', OUTPUT_DRIVE_DIR)